# Session 8 — Gradio Chat Demo

This notebook shows how to wrap an OpenAI SDK call in a Gradio UI — first as a simple request/response app, then upgraded to a streaming interface.

## Learning Goals

- Connect a model call to a simple UI
- Understand how Gradio passes conversation history to your function
- Reuse the streaming generator pattern to make the UI feel responsive
- Understand why Gradio is useful for quick demos

In [64]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_ORG_ID = os.getenv('OPENAI_ORG_ID')
OPENAI_PROJECT_ID = os.getenv('OPENAI_PROJECT_ID')

print('OpenAI key present:', bool(OPENAI_API_KEY))
print('OpenAI org ID present:', bool(OPENAI_ORG_ID))
print('OpenAI project ID present:', bool(OPENAI_PROJECT_ID))

OpenAI key present: True
OpenAI org ID present: True
OpenAI project ID present: True


## Part 1 — Basic Chat

[Gradio](https://www.gradio.app/) is a Python library for building quick web UIs around ML models and APIs. `gr.ChatInterface` is its high-level helper specifically for chatbots: you give it a Python function and it handles the chat window, input box, message rendering, and a launchable local web server.

**What your function must look like:**

```python
def fn(message: str, history: list[dict]) -> str:
    ...
```

- `message` — the user's latest input as a plain string
- `history` — a list of previous turns, each a dict with `role` and `content`:

```python
[
  {'role': 'user',      'content': [{'type': 'text', 'text': 'Hello'}]},
  {'role': 'assistant', 'content': [{'type': 'text', 'text': 'Hi there!'}]},
]
```

Gradio manages the history for you — your job is to unpack it into whatever format the model API expects, call the model, and return the reply as a string.

In [65]:
def on_message_submit(message, history):

    # Gradio passes history as a list of role/content dicts.
    # We unpack it into the flat format the OpenAI Responses API expects.
    messages = []

    print('incoming history:', history)

    for item in history:
        messages.append({'role': item['role'], 'content': item['content'][0]['text']})

    # Append the new user message
    messages.append({'role': 'user', 'content': message})

    print('updated messages history:', messages)

    client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)

    # Call the model and return the full response text
    response = client.responses.create(
        model='gpt-4o-mini',
        input=messages
    )

    return response.output_text

In [ ]:
gr.ChatInterface(
    on_message_submit,
    title='Simple Gradio Chatbot',
    description='A minimal chat app using the OpenAI SDK.'
).launch()

* Running on local URL:  http://127.0.0.1:7893
* To create a public link, set `share=True` in `launch()`.


incoming history: []
updated messages history: [{'role': 'user', 'content': 'Hello! '}]
incoming history: [{'role': 'user', 'metadata': None, 'content': [{'text': 'Hello! ', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': 'Hello! How can I assist you today?', 'type': 'text'}], 'options': None}]
updated messages history: [{'role': 'user', 'content': 'Hello! '}, {'role': 'assistant', 'content': 'Hello! How can I assist you today?'}, {'role': 'user', 'content': 'What is the capital of france? '}]


## Part 2 — Streaming

The basic version above waits for the model to finish generating before showing anything. For short replies that's fine, but for longer answers the UI just hangs — bad UX.

**The fix:** use a streaming response and `yield` partial text as it arrives. Gradio detects that your function is a generator and automatically updates the chat window with each yielded chunk — no extra wiring needed.

Two things change relative to the basic version:

1. `stream=True` on the API call
2. `return response.output_text` → `yield partial` inside an event loop

The event type to watch for is `response.output_text.delta`, which carries each new piece of text. We accumulate the chunks into `partial` and yield after every update so Gradio can render incrementally.

In [67]:
def stream_chat_response(message, history):
    client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)

    # Same history unpacking as before
    messages = []

    print('history type:', type(history))
    print('history before processing:', history)

    for item in history:
        messages.append({'role': item['role'], 'content': item['content'][0]['text']})
    messages.append({'role': 'user', 'content': message})

    print('history after processing:', messages)

    # stream=True returns an iterator of server-sent events
    stream = client.responses.create(
        model='gpt-4o-mini',
        input=messages,
        stream=True,
    )

    partial = ''
    for event in stream:
        # response.output_text.delta carries each new text chunk
        if event.type == 'response.output_text.delta':
            partial += event.delta
            yield partial  # Gradio re-renders the message bubble on every yield

In [68]:
demo = gr.ChatInterface(
    fn=stream_chat_response,
    title='Session 8 Gradio Demo - with streaming',
    description='A minimal streaming chat app using the OpenAI SDK.'
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7894
* To create a public link, set `share=True` in `launch()`.


history type: <class 'list'>
history before processing: []
history after processing: [{'role': 'user', 'content': 'hello! '}]
history type: <class 'list'>
history before processing: [{'role': 'user', 'metadata': None, 'content': [{'text': 'hello! ', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': 'Hello! How can I assist you today?', 'type': 'text'}], 'options': None}]
history after processing: [{'role': 'user', 'content': 'hello! '}, {'role': 'assistant', 'content': 'Hello! How can I assist you today?'}, {'role': 'user', 'content': 'what is the capital of France? '}]
history type: <class 'list'>
history before processing: [{'role': 'user', 'metadata': None, 'content': [{'text': 'hello! ', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': 'Hello! How can I assist you today?', 'type': 'text'}], 'options': None}, {'role': 'user', 'metadata': None, 'content': [{'text': 'what is the capital of F

## What Changed?

| | Basic | Streaming |
|---|---|---|
| Return type | `return str` | `yield partial_str` |
| API call | `stream=False` (default) | `stream=True` |
| When does text appear? | After full response | Incrementally, chunk by chunk |
| Gradio wiring needed | None | None — Gradio detects a generator automatically |

Both functions have the same signature: `fn(message, history)`. Gradio doesn't care whether your function returns or yields — it handles both transparently.